# giskard × Isaac Sim: Stretch control demo

Drive the Stretch robot in the apartment scene through
[giskardpy](https://github.com/cram2/cognitive_robot_abstract_machine)'s
closed-loop whole-body QP controller: joint-space, Cartesian end-effector,
base, and gripper goals.

**Kernel**: select **CRAM** (registered by the Docker image; register manually by
copying `binder/cram_python_wrapper.sh` to `~/.local/bin/` and
`binder/cram-kernel.json` to `~/.local/share/jupyter/kernels/cram/kernel.json`).

The two cells below start the simulation and the giskard server as background
processes. Skip them if you already started these in a terminal. They kill stale
instances from previous runs first: duplicated simulations/servers fight
over the same topics and make the robot oscillate.


## Start the Isaac Sim simulation

First startup can take a few minutes (shader compilation). Progress is written
to `/tmp/isaac_sim.log`.


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parent  # this notebook lives in demos/
sys.path.insert(0, str(REPO))

from launcher import start_isaac_sim, start_giskard_server, start_rviz, stop

# import os
# os.environ["DISPLAY"] = ":0"

rviz_proc = start_rviz()          # terminal=True to open a gnome-terminal window
sim_proc = start_isaac_sim(camera="both")          # terminal=True to open a gnome-terminal window
giskard_proc = start_giskard_server()

## Connect to giskard

Fetches the world model from the server and defines a small execution helper.
`add_end_conditions` holds a reached goal for one extra second before ending the
motion (settle phase): giskard's behavior tree keeps publishing the last commanded
velocities for about a second between goal completion and its terminate-zero
message, so ending immediately lets the base coast past the goal.


In [ ]:
from cram_vrb_lab.control.giskard_client import add_end_conditions, add_external_collision_avoidance, connect
from giskardpy.motion_statechart.data_types import ObservationStateValues
from giskardpy.motion_statechart.motion_statechart import MotionStatechart

giskard = connect('giskard_notebook_client')
world = giskard.world
print('connected, robot:', giskard.robot_name)


def run_goal(task, timeout=60.0, avoid_collisions=True):
    """Execute a single motion task with settle phase + timeout.

    avoid_collisions adds whole-body external collision avoidance, so the robot
    keeps a safety margin from the apartment walls and furniture that
    the giskard server loads into the world. Set it False to reproduce the
    old straight-line behaviour that drives through furniture.
    """
    msc = MotionStatechart()
    msc.add_node(task)
    if avoid_collisions:
        add_external_collision_avoidance(msc)
    add_end_conditions(msc, task, timeout_seconds=timeout)
    giskard.execute(msc)
    reached = msc.observation_state[task] == ObservationStateValues.TRUE
    print('goal reached:', reached)
    return reached


## 1. Joint-space goal

Command joint target positions directly. Available joints: see
`cram_vrb_lab.robots.stretch.joints.CONTROLLED_JOINTS`. Threshold 0.02: this pipeline settles within
0.01-0.02 of the goal, so the default 0.01 latches unreliably.


In [ ]:
from giskardpy.motion_statechart.tasks.joint_tasks import JointPositionList
from semantic_digital_twin.datastructures.joint_state import JointState

goal = {
    world.get_connection_by_name('joint_lift'): 0.2,
    world.get_connection_by_name('joint_wrist_yaw'): 0.0,
}
run_goal(JointPositionList(goal_state=JointState.from_mapping(goal), threshold=0.02),
         timeout=30.0)


## 2. Gripper open / close


In [ ]:
FINGER_OPEN, FINGER_CLOSED = 0.55, 0.0


def gripper(position):
    goal = {
        world.get_connection_by_name('joint_gripper_finger_left'): position,
        world.get_connection_by_name('joint_gripper_finger_right'): position,
    }
    return run_goal(JointPositionList(goal_state=JointState.from_mapping(goal),
                                      threshold=0.02), timeout=30.0)


gripper(FINGER_OPEN)


## 3. Cartesian end-effector goal (arm only)

Translate `link_grasp_center` in its own frame, keeping the current orientation.
`root_link=base_link` restricts the motion to the arm (base stays put).
Note: self-collision avoidance is not enabled -- avoid goals that press the
gripper into the robot's own body.


In [ ]:
from giskardpy.motion_statechart.tasks.cartesian_tasks import CartesianPose
from semantic_digital_twin.spatial_types.spatial_types import Pose, Vector3

tip = world.get_kinematic_structure_entity_by_name('link_grasp_center')
base = world.get_kinematic_structure_entity_by_name('base_link')

goal_pose = Pose(position=Vector3(0.1, 0.0, 0.15), reference_frame=tip)  # 15 cm up
run_goal(CartesianPose(root_link=base, tip_link=tip, goal_pose=goal_pose, threshold=0.03))


## 4. Cartesian end-effector goal (whole body)

With `root_link=world.root` the base joins the motion -- if the target is out of
the arm's reach, the base drives to make up the difference.

`CartesianPose` parameters:

- **`root_link`** -- the *base* of the kinematic chain giskard is allowed to move.
  Only joints between `root_link` and `tip_link` are actuated: `base_link` means
  arm only, `world.root` means the base joins in.
- **`tip_link`** -- the link that has to reach the goal.
- **`goal_pose`** -- the target, expressed in `goal_pose.reference_frame`. **This
  frame is what the numbers are relative to, not `root_link` and not `base_link`.**
  With `reference_frame=tip` the pose is relative to the gripper's *own* current
  frame, so `Vector3(0.4, -1.0, 0.0)` means "0.4 m along the gripper's x, 1.0 m
  against its y". The frame is snapshotted by forward kinematics once, when the
  task starts (`binding_policy=Bind_on_start`), and then held fixed in `root_link`
  coordinates -- it is a fixed target, not one that runs away as the tip moves.
- **`goal_pose`'s orientation** defaults to identity. Identity *relative to the
  reference frame* -- which is why `reference_frame=tip` keeps the current
  orientation, while `reference_frame=base` would demand the gripper align with
  `base_link`'s axes.
- **`threshold`** -- applies to both errors: metres for position, radians for
  rotation. The goal counts as reached only when both are below it.
- **`timeout`** is `run_goal`'s own argument (see `add_end_conditions`), not part
  of the task.


In [ ]:
goal_pose = Pose(position=Vector3(-1, 1, 0.3), reference_frame=tip)
run_goal(CartesianPose(root_link=world.root, tip_link=tip, goal_pose=goal_pose,
                       threshold=0.03), timeout=90.0)


### Commanding an orientation

`Pose(orientation=...)` takes a `Quaternion`; build it from roll/pitch/yaw with
`Quaternion.from_rpy` or from an axis with `Quaternion.from_axis_angle`. The
reference frame decides whether it is a *relative* or an *absolute* rotation:

- `reference_frame=tip` -> rotate the gripper by that much **from where it is now**
- `reference_frame=base` -> put the gripper into that orientation **in the base
  frame**, whatever it is doing right now


In [ ]:
import math

from semantic_digital_twin.spatial_types.spatial_types import Quaternion

# Relative: lift 10 cm and roll the wrist 90 deg about the gripper's own z axis,
# arm only (root_link=base). Both numbers are read in the tip frame.
goal_pose = Pose(position=Vector3(0.0, 0.0, 0.1),
                 orientation=Quaternion.from_rpy(0.0, 0.0, math.pi / 2),
                 reference_frame=tip)
run_goal(CartesianPose(root_link=base, tip_link=tip, goal_pose=goal_pose,
                       threshold=0.03), timeout=60.0)


In [ ]:
# Absolute: a fixed spot 0.5 m in front of where base_link is *now*, 0.9 m up,
# gripper aligned with the base axes -- whatever pose it currently has.
# Whole body (root_link=world.root): Stretch's arm telescopes sideways, so a
# target straight ahead is out of reach unless the base is allowed to turn.
# The reference frame is snapshotted at task start, so the goal stays put in the
# world even though the base moves.
goal_pose = Pose(position=Vector3(0.5, 0.0, 0.9),
                 orientation=Quaternion.from_rpy(0.0, 0.0, 0.0),
                 reference_frame=base)
run_goal(CartesianPose(root_link=world.root, tip_link=tip, goal_pose=goal_pose,
                       threshold=0.03), timeout=90.0)


### Goals in world coordinates

`reference_frame=world.root` gives a fixed target in the world, independent of
where the robot is. `world.root` is the body named `map`; the kinematic chain is
`map -(localization, 6DoF)-> odom -(differential drive)-> base_link -> ... -> tip`.

In *this* setup those are literally Isaac Sim stage coordinates: the sim
publishes the base link's absolute stage transform on `/odom` (`StretchROS.publish_odom` in
`cram_vrb_lab.robots.stretch.isaac_node`), and the server runs no SLAM, so the `map -> odom` localization
joint stays identity (the static `map -> odom` stand-in) -- `map` == `odom` == the
USD `/World` origin. The apartment prim sits at `(-6, 5, 0.07)` and the robot
spawns at `(-1.5, 0, 0.05)`, so pick world targets in that neighbourhood.

On a real robot with localization running this would no longer hold: `map` would
be the SLAM frame and `odom` would drift away from it.


In [ ]:
# Where is the gripper right now, in world coordinates?
map_T_tip = world.compute_forward_kinematics(world.root, tip)
print('world.root  :', world.root.name)
print('tip in world:', map_T_tip.to_position().to_np().round(3))


In [ ]:
# Absolute world target: whole body, so the base drives there if the arm can't
# reach. Run the cell above first and pick a point within ~1 m of the gripper.
# There is no path planning here -- the QP pulls in a straight line -- but
# run_goal now adds collision avoidance by default, so the robot brakes near the
# apartment furniture instead of driving through it. Pass avoid_collisions=False
# to get the old push-through-anything behaviour.
goal_pose = Pose(position=Vector3(-1.0, 0.1, 0.5),
                 orientation=Quaternion.from_rpy(0.0, 0.0, 90.0),
                 reference_frame=world.root)
run_goal(CartesianPose(root_link=world.root, tip_link=tip, goal_pose=goal_pose,
                       threshold=0.1), timeout=90.0)


## 5. Base goal

`DifferentialDriveBaseGoal` (orient -> drive -> orient) is the right idiom for a
non-holonomic base; a plain `CartesianPose` on `base_link` oscillates around the
goal. The 5 cm threshold is the usual tolerance for a mobile base.


In [ ]:
from giskardpy.motion_statechart.goals.cartesian_goals import DifferentialDriveBaseGoal

goal_pose = Pose(position=Vector3(-0.2, 0.0, 0.0), reference_frame=base)  # 2 m backwards
run_goal(DifferentialDriveBaseGoal(goal_pose=goal_pose, threshold=0.05))


## 6. Collision avoidance with the apartment

`run_goal(..., avoid_collisions=True)` (now the default) adds
`ExternalCollisionAvoidance`: the whole-body QP keeps a safety margin from every
collision body giskard knows about -- the apartment walls and furniture that
`stretch_apartment_giskard_server.py` loads into the world via
`WorldWithStretchAndApartmentDiffDrive`.

Drive the base toward the apartment and watch it steer / brake around obstacles.
Re-run the same goal with `avoid_collisions=False` to compare: the QP does no path
planning, so without avoidance it drives straight through the furniture.

If the robot refuses to move, it may already start inside the *violated* distance
of a mis-aligned collision body (`ExternalCollisionAvoidance` cancels the motion in
that case). Tune `USD_PRIM_POSITION_IN_MAP` in `cram_vrb_lab/scenes/apartment/constants.py` so the loaded
apartment matches the Isaac scene, or relax it with a custom
`add_external_collision_avoidance(msc, cancel_if_collision_violated=False)`.


In [ ]:
# Whole-body base drive toward the apartment. avoid_collisions=True (the default)
# keeps a safety margin from the walls and furniture; flip it to False to watch
# the robot drive straight through them for comparison.
goal_pose = Pose(position=Vector3(-3, 0.0, 0.0), reference_frame=base)  # 1.5 m ahead
run_goal(DifferentialDriveBaseGoal(goal_pose=goal_pose, threshold=0.05),
         timeout=90.0, avoid_collisions=True)


: 

### Escape: recover from a collision-violated deadlock

Once the robot stands inside the *violated* distance of a collision body,
**every** goal that includes the default collision avoidance is cancelled
immediately (`CollisionViolatedError`) -- the watchdog only checks "is any pair
violated right now", not whether the motion would make things better. Even a
retreat goal throws.

`escape()` backs the base up with `cancel_if_collision_violated=False`: the
watchdog is off, but the avoidance *constraints* stay active, and since they are
one-sided ("increase the distance"), they actively push the robot out of the
violated zone while it retreats. If the world model is so mis-aligned that even
this fails, call `escape(keep_avoidance=False)` for a blind retreat.

In [ ]:
def escape(distance=0.4, timeout=30.0, keep_avoidance=True):
    """Back the base up without the collision-violated watchdog, to get out of
    a state where every normal goal is cancelled at t=0."""
    task = DifferentialDriveBaseGoal(
        goal_pose=Pose(position=Vector3(-distance, 0.0, 0.0), reference_frame=base),
        threshold=0.05,
    )
    msc = MotionStatechart()
    msc.add_node(task)
    if keep_avoidance:
        add_external_collision_avoidance(msc, cancel_if_collision_violated=False)
    add_end_conditions(msc, task, timeout_seconds=timeout)
    giskard.execute(msc)
    reached = msc.observation_state[task] == ObservationStateValues.TRUE
    print('escaped:', reached)
    return reached


# escape()  # uncomment to back out of a collision-violated deadlock

## Shutdown

Stop the giskard server and the simulation (only if they were started from this
notebook).


In [ ]:
# stop()  # stops the isaac sim + giskard server started above


## Troubleshooting

- **Gripper stuck after touching an object**: reset it through the sim's native
  interface: `ros2 topic pub --once /stretch/gripper_command std_msgs/msg/Float64 "{data: 0.05}"`
- **Arm in a twisted pose**: send a giskard joint goal (a `JointPositionList` as in
  section 1) to move it back to a neutral configuration
- giskard is a whole-body controller: even an arm-only goal may slightly adjust
  other controlled joints
- More details in `demos/README.md`
